In [3]:
from helper.config import load_config
import psycopg2
from datetime import datetime
import pandas as pd
import numpy as np

from_date = 20241101
to_date = 20241101

# Convert dates to UNIX timestamps
from_date_unix = int(datetime.strptime(str(from_date) + ' 00:00:00', '%Y%m%d %H:%M:%S').timestamp()) + 25200 # Add 7 hours
to_date_unix = int(datetime.strptime(str(to_date) + ' 23:59:59', '%Y%m%d %H:%M:%S').timestamp()) + 25200 # Add 7 hours

config = load_config()
with psycopg2.connect(**config) as conn:
    with conn.cursor() as cur:
            cur.execute(f"""
                    select 
                        a.start_time
                        ,a.end_time
                        ,c.lat as start_lat
                        ,c.lon as start_lon
                        ,d.lat as end_lat
                        ,d.lon as end_lon
                        ,a.vehicle_type
                        ,a.distance_meters
                        ,c.location_name as start_location_name
                        ,c.category as start_category
                        ,d.location_name as end_location_name
                        ,d.category as end_category
                        ,json_agg(json_build_object('txtime', b.txtime , 'lat', b.lat, 'lon', b.lon)) AS path
                    from gmap.fact_activity a
                    cross join gmap.fact_timelinepath b
                    left join gmap.dim_location c
                    on a.start_location_id = c.location_id
                    left join gmap.dim_location d
                    on a.end_location_id = d.location_id
                    where a.start_time between {from_date_unix} AND {to_date_unix}
                    and b.txtime between a.start_time and a.end_time 
                    group by 
                        a.start_time
                        ,a.end_time
                        ,c.lat
                        ,c.lon
                        ,d.lat
                        ,d.lon
                        ,a.vehicle_type
                        ,a.distance_meters
                        ,c.location_name
                        ,c.category
                        ,d.location_name
                        ,d.category
                        """)
            activity = pd.DataFrame(cur.fetchall(), columns=[desc[0] for desc in cur.description]) 

            cur.execute(f"""
                    select 
                        a.start_time 
                        ,a.end_time 
                        ,c.lat 
                        ,c.lon
                        ,c.location_name
                        ,c.category
                        ,json_agg(json_build_object('txtime', b.txtime , 'lat', b.lat, 'lon', b.lon)) AS path
                    from gmap.fact_visit a
                    cross join gmap.fact_timelinepath b
                    left join gmap.dim_location c
                    on a.location_id = c.location_id
                    where a.start_time between {from_date_unix} AND {to_date_unix}
                    and b.txtime between a.start_time and a.end_time 
                    group by 
                        a.start_time 
                        ,a.end_time 
                        ,c.lat 
                        ,c.lon
                        ,c.location_name
                        ,c.category
                        """)
            visit = pd.DataFrame(cur.fetchall(), columns=[desc[0] for desc in cur.description]) 

# add info to activity
activity['path'] = activity['path'].apply(lambda x: {item['txtime']: [item['lat'], item['lon']] for item in x})
activity['color'] = activity['vehicle_type'].map({
    'in bus': 'red',
    'in passenger vehicle': 'blue',
    'walking': 'green',
    'motorcycling': 'yellow',
    'unknown': 'black'
})
activity['start_time_human'] = pd.to_datetime(activity['start_time'], unit='s')
activity['end_time_human'] = pd.to_datetime(activity['end_time'], unit='s')

# add info to visit
visit['path'] = visit['path'].apply(lambda x: {item['txtime']: [item['lat'], item['lon']] for item in x})
visit['duration'] = visit['end_time'] - visit['start_time']
visit['duration'] = visit['duration'].apply(lambda x: '{} hours {} minutes'.format(int(divmod(x, 60*60)[0]), int(divmod(divmod(x, 60*60)[1], 60)[0])))
visit['start_time_human'] = pd.to_datetime(visit['start_time'], unit='s')
visit['end_time_human'] = pd.to_datetime(visit['end_time'], unit='s')

# Get all coordinates for map zoom display
visit_path_coordinates = sum(visit['path'].apply(lambda x: list(x.values() if isinstance(x, dict) else [])),[])
visit_coordinates = visit[['lat', 'lon']].apply(lambda x: [float(x['lat']), float(x['lon'])], axis=1).to_list()
activity_path_coordinates = sum(activity['path'].apply(lambda x: list(x.values() if isinstance(x, dict) else [])),[])
activity_start_coordinates = activity[['start_lat', 'start_lon']].apply(lambda x: [float(x['start_lat']), float(x['start_lon'])], axis=1).to_list()
activity_end_coordinates = activity[['end_lat', 'end_lon']].apply(lambda x: [float(x['end_lat']), float(x['end_lon'])], axis=1).to_list()
coordinates = sum([visit_path_coordinates, visit_coordinates, activity_path_coordinates, activity_start_coordinates, activity_end_coordinates], [])
min_lat = min(coordinates, key=lambda x: x[0])[0]
max_lat = max(coordinates, key=lambda x: x[0])[0]
min_lon = min(coordinates, key=lambda x: x[1])[1]
max_lon = max(coordinates, key=lambda x: x[1])[1]

pd.DataFrame(
      np.concatenate([
      activity[['start_time_human', 'end_time_human', 'start_time', 'end_time', 'start_location_name', 'end_location_name', 'vehicle_type']].to_numpy(), 
      visit[['start_time_human', 'end_time_human', 'start_time', 'end_time', 'location_name', 'location_name', 'location_name']].to_numpy()
      ])
      , columns=['start_time_human', 'end_time_human', 'start_time', 'end_time', 'start_location_name', 'end_location_name', 'vehicle_type']
      ).sort_values('start_time')


,start_time_human,end_time_human,start_time,end_time,start_location_name,end_location_name,vehicle_type
0,2024-11-01 08:06:32,2024-11-01 08:38:35,1730448392,1730450315,Siêu thị FujiMart Ngọc Khánh,39 Khuất Duy Tiến,in passenger vehicle
6,2024-11-01 08:38:36,2024-11-01 08:40:36,1730450316,1730450436,162 Khuất Duy Tiến,162 Khuất Duy Tiến,162 Khuất Duy Tiến
5,2024-11-01 08:40:37,2024-11-01 11:26:53,1730450437,1730460413,Work,Work,Work
4,2024-11-01 11:26:54,2024-11-01 12:10:51,1730460414,1730463051,Bún cá,Bún cá,Bún cá
1,2024-11-01 12:10:52,2024-11-01 12:13:38,1730463052,1730463218,Bún cá,39 Khuất Duy Tiến,walking
9,2024-11-01 12:13:39,2024-11-01 18:03:31,1730463219,1730484211,Work,Work,Work
2,2024-11-01 18:03:32,2024-11-01 18:19:40,1730484212,1730485180,39 Khuất Duy Tiến,162 Khuất Duy Tiến,walking
8,2024-11-01 18:19:41,2024-11-01 18:29:39,1730485181,1730485779,162 Khuất Duy Tiến,162 Khuất Duy Tiến,162 Khuất Duy Tiến
10,2024-11-01 18:29:40,2024-11-01 18:38:32,1730485780,1730486312,Siêu thị FujiMart Ngọc Khánh,Siêu thị FujiMart Ngọc Khánh,Siêu thị FujiMart Ngọc Khánh
3,2024-11-01 18:38:33,2024-11-01 19:02:20,1730486313,1730487740,Siêu thị FujiMart Ngọc Khánh,807 Giải Phóng,in passenger vehicle


In [4]:
import folium
from folium.plugins import MarkerCluster, AntPath

f = folium.Figure(width=1000, height=600)
m = folium.Map(
                location=[(min_lat + max_lat)/2, (min_lon + max_lon)/2],
                zoom_start=13, 
                control_scale=True,
                tiles="cartodbpositron",
               ).add_to(f)

# # if the points are too close to each other, cluster them, create a cluster overlay with MarkerCluster
marker_cluster = MarkerCluster().add_to(m)

for _, item in activity.iterrows():
    # print(item['path'])
    tooltip_txt = '<p>'\
                +'From: '\
                +pd.to_datetime(item['start_time'], unit = 's').strftime('%Y-%m-%d %H:%M:%S')\
                +'<br> To: '\
                +pd.to_datetime(item['end_time'], unit = 's').strftime('%Y-%m-%d %H:%M:%S')\
                +'<br> Transportation: '\
                +item['vehicle_type']\
                +'</p>'
    if isinstance(item['path'], dict):
        AntPath(
            locations=list(item['path'].values()), 
            color=item['color'],
            delay=800,
            weight=5,
            opacity=0.5,
            # reverse="True", 
            dash_array=[10, 20],
            tooltip=tooltip_txt
        ).add_to(m)

    folium.Marker(
                    location = (item['start_lat'], item['start_lon']),
                    icon = folium.Icon(icon='play', prefix='glyphicon', color='orange'),
                    tooltip=tooltip_txt
                ).add_to(marker_cluster)    
    
    folium.Marker(
                    location = (item['end_lat'], item['end_lon']),
                    icon = folium.Icon(icon='stop', prefix='glyphicon', color='blue'),
                    tooltip=tooltip_txt
                ).add_to(marker_cluster)    
        
for _, item in visit.iterrows():
    if isinstance(item['path'], dict):
        folium.PolyLine(
            list(item['path'].values()), 
            color='black',
            weight=2,
            opacity=0.5,
            # dash_array='5, 5'
            ).add_to(m)
    
    tooltip_txt = '<p>'\
                +'From: '\
                +pd.to_datetime(item['start_time'], unit = 's').strftime('%Y-%m-%d %H:%M:%S')\
                +'<br> To: '\
                +pd.to_datetime(item['end_time'], unit = 's').strftime('%Y-%m-%d %H:%M:%S')\
                +'<br> Duration: '\
                +item['duration']\
                +'</p>'
    
    folium.Marker(
                    location = (item['lat'], item['lon']),
                    icon = folium.Icon(icon='ok', prefix='glyphicon', color='green'),
                    tooltip = tooltip_txt
                ).add_to(marker_cluster)    
    
m.fit_bounds(m.get_bounds())

m